# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset loaded: {dataset.metadata.name}\n")
print(dataset.metadata.description)

## 2. Data Overview
Review available record sets and fields. All entities are referenced via their `@id`.

Let's list the record sets, each field available in them, and get their labels and `@id`s.

In [ ]:
# List all record sets and their fields with @id
record_set_summaries = []

for rs in dataset.record_sets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}, dataType: {f.data_type})")
    record_set_summaries.append(rs.id)
    print("")
if not record_set_summaries:
    print("No RecordSets found.\nPlease check the Croissant schema for updates.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the previous overview.

In [ ]:
# Prepare to extract data from discovered record sets
if not dataset.record_sets:
    raise RuntimeError("No RecordSets defined in the schema.")

# For this dataset, let's use the first available record set
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    # Load records from the record set specified by @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Display summary
for record_set_id, df in dataframes.items():
    print(f"RecordSet: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records by a numeric field, normalizing, and grouping.

--
**Note:** For demonstration, we choose the first numeric field found in the main RecordSet. If you know specific field `@id`s, set them explicitly.

In [ ]:
# Pick the first available record set
chosen_rs_id = record_sets[0]
df = dataframes[chosen_rs_id]

# Identify a numeric field by field @id and dataType
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets:
    if rs.id == chosen_rs_id:
        # Try to find an integer/float/number field
        for f in rs.fields:
            if (f.data_type or '').lower() in ["integer", "number", "float"]:
                numeric_field_id = f.id
                break
        # Try to find a text/categorical field for grouping
        for f in rs.fields:
            if (f.data_type or '').lower() in ["text", "string"]:
                group_field_id = f.id
                break
        break

if numeric_field_id is None:
    raise RuntimeError("No numeric field found in the main record set.")

print(f"Numeric field chosen for analysis: {numeric_field_id}")
if group_field_id:
    print(f"Grouping field: {group_field_id}")

# Filter: Only use values where the numeric field > its median
if numeric_field_id not in df.columns:
    raise RuntimeError(f"Field {numeric_field_id} not in dataframe columns.")

# Sometimes Croissant parsing yields all str - convert to numeric cleanly
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
median_value = df[numeric_field_id].median()
filtered_df = df[df[numeric_field_id] > median_value]
print(f"Filtered records with {numeric_field_id} > median ({median_value}): {len(filtered_df)} records")
display(filtered_df.head())

# Normalize the chosen numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the chosen group field if available
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field, and optionally the normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=25, kde=True)
plt.title(f"Distribution of {numeric_field_id} (filtered > median)")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Visualize normalized field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=25, kde=True, color='orange')
plt.title(f"Distribution of Normalized {numeric_field_id} (filtered)")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.ylabel("Count")
plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset's mass spectrometry data, explored its schema, extracted main record set data using `mlcroissant`, and performed quick EDA and visualization. You can now adapt this notebook to deeper domain-specific analyses, such as analyzing distributions across proteins, exploring modifications, or integrating metadata from additional linked fields as referenced by `@id` in the Croissant schema.

- Explore additional record sets and fields using their `@id`.
- Use the data dictionary and Croissant JSON-LD for more context and rich analyses.
- See [mlcroissant documentation](https://mlcommons.github.io/croissant/python/latest/) for advanced querying options.